In [1]:
# Install Ollama and Python dependencies
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install jiwer tabulate ollama -q

print("Setup Complete!")

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 129802 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%################                                               37.7%                    74.2%
>>> Creating ollama user

In [2]:
import os, time, subprocess

# Kill any existing server
!pkill -9 ollama
time.sleep(2)

# Start Ollama Server in Background
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Download the vision model
!ollama pull glm-ocr
print("Model ready!")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 65493e1f85b9:   0% ▕                  ▏ 2.9 MB/2.2 GB                  pulling manifest 
pulling 65493e1f85b9:   2% ▕                  ▏  42 MB/2.2 GB                  pulling manifest 
pulling 65493e1f85b9:   6% ▕█                 ▏ 137 MB/2.2 GB                  pulling manifest 
pulling 65493e1f85b9:  11% ▕█                 ▏ 239 MB/2.2 GB                  pulling manifest 
pulling 65493e1f85b9:  13% ▕██                ▏ 282 MB/2.2 GB                  pulling manifest 
pulling 65493e1f85b9:  17% ▕███               ▏ 370 MB/2.2 GB                  pulling manifest 
pulling 65493e1f85b9:  21% ▕███               ▏ 456 MB/2.2 GB        

In [3]:
modelfile_content = """FROM glm-ocr

PARAMETER num_ctx 8192
PARAMETER temperature 0.0

SYSTEM \"\"\"
You are an expert OCR AI specialized in reading handwritten warehouse Credit Note Requisition (CNR) forms.
Your task is to extract only the handwritten text and output it as a strict JSON object.

CRITICAL RULES:
1. ONLY EXTRACT HANDWRITING. Ignore pre-printed labels.
2. CUSTOMER NAME: Carefully read the handwriting next to "CUSTOMER NAME".
3. INVOICE/DO NO: Read the invoice number if written.
4. ITEMS TABLE: Extract the rows of handwritten items. For 'qty', include the unit if written. For 'description', read the handwriting carefully.

You MUST return exactly this JSON structure, with no duplicate keys and no missing keys:
{
    "cnr_no": "",
    "date": "",
    "location": "",
    "prepared_by": "",
    "customer_name": "",
    "invoice_do_no": "",
    "items": [
        {
            "qty": "",
            "description": "",
            "packing": "",
            "unit_price_rm": null,
            "reason_code": "",
            "wh_location": "",
            "remarks": ""
        }
    ]
}
Return ONLY valid JSON. Do not include markdown formatting like ```json or any extra text.
\"\"\""""

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# Build the custom model
!ollama create cnr-reader -f Modelfile
print("Model Built!")

]11;?\gathering model components 
using existing layer sha256:65493e1f85b9ea4ba3ed793515fde13cbdbea7d74ad2c662b566b146eab0081e 
creating new layer sha256:218e51585b48ade9db74ca733a1d9416a35fa7f488fb7d5d56b40cd99a0f4c62 
creating new layer sha256:fc258d6aa9b979530ec86c8531955e2a87dcd852f63c9eaf10ce7ea5a0538c28 
writing manifest ⠋ gathering model components 
using existing layer sha256:65493e1f85b9ea4ba3ed793515fde13cbdbea7d74ad2c662b566b146eab0081e 
creating new layer sha256:218e51585b48ade9db74ca733a1d9416a35fa7f488fb7d5d56b40cd99a0f4c62 
creating new layer sha256:fc258d6aa9b979530ec86c8531955e2a87dcd852f63c9eaf10ce7ea5a0538c28 
writing manifest 
success 
Model Built!


In [ ]:
import base64
import json
import re
import os
from ollama import Client
import jiwer
from tabulate import tabulate

# Update this dataset path
DATASET_PATH = "/kaggle/input/datasets/lexivoo/actual-cnr-samples/Client_CNR_Dataset"
OLLAMA_CLIENT = Client(host='http://127.0.0.1:11434')

# Formats text for mathematical comparison (removes punctuation, fixes 1.0 float)
def text_normalization(text):
    text = str(text).strip().lower()
    if text.endswith('.0'):
        text = text[:-2] 
    cleaned_text = re.sub(r'[^a-z0-9\s]', ' ', text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text if cleaned_text else "empty"

# Removes letters and symbols
def extract_numbers_only(text):
    return re.sub(r'[^0-9]', '', str(text))

def calculate_word_precision(gt_text, ai_text):
    if gt_text == "empty" and ai_text == "empty":
        return 100.0
    if gt_text == "empty" or ai_text == "empty":
        return 0.0
        
    gt_words = set(gt_text.split())
    ai_words = set(ai_text.split())
    correct_words = len(ai_words.intersection(gt_words))
    precision = (correct_words / len(ai_words)) * 100
    return precision
    

def call_ai_model(image_base64):
    system_prompt = '''Extract the data from this form into JSON.
        You MUST follow these strict rules:
        1. ONLY extract handwritten text (blue/black ink).
        2. "invoice_do_no" is the handwritten number inside the box.
        3. DO NOT leave "customer_name" blank. Look for the thick, black cursive handwriting directly to the right of the "CUSTOMER NAME" label.
        4. For description, read carefully.
        5. "cnr_no" CAN be left completely blank ("").'''
    try:
        response = OLLAMA_CLIENT.chat(
            model='cnr-reader',
            format='json',
            messages=[{
                'role': 'user',
                'content': system_prompt,
                'images': [image_base64]
            }],
            options={
                'temperature': 0.0,
                'num_ctx': 8192,
                'num_predict': 400,
                'stop': ['```', '---']
            }
        )
        raw_output = response.get('message', {}).get('content', '').strip()
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
            
    except Exception:
        pass 
        
    return {} 

    
def evaluate_model(dataset_folder_name):
    print(f"\nEvaluating: {dataset_folder_name.upper()} COPIES")
    
    image_dir = os.path.join(DATASET_PATH, dataset_folder_name, "images")
    annotation_dir = os.path.join(DATASET_PATH, "annotations")
    
    total_fields_scored = 0         
    total_numeric_fields = 0        
    total_exact_matches = 0
    total_accuracy_score = 0.0
    
    if not os.path.exists(image_dir):
        print(f"Error: Could not find {image_dir}")
        return 0, 0
        
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.png')]
    total_samples = len(image_files)
    
    for index, filename in enumerate(image_files, 1):
        cnr_id = filename.split('.')[0]
        json_path = os.path.join(annotation_dir, f"{cnr_id}.json")
        
        if not os.path.exists(json_path):
            continue
            
        with open(json_path, 'r') as file:
            ground_truth = json.load(file)
    
        with open(os.path.join(image_dir, filename), "rb") as image_file:
            img_b64 = base64.b64encode(image_file.read()).decode("utf-8")
            
        print(f"[{index}/{total_samples}] Extracting CNR No {cnr_id}...")
        ai_prediction = call_ai_model(img_b64)
            
        numeric_fields = []
        text_fields = []
        
        gt_invoice = extract_numbers_only(ground_truth.get('invoice_do_no', ''))
        ai_invoice = extract_numbers_only(ai_prediction.get('invoice_do_no', ''))
        numeric_fields.append((gt_invoice, ai_invoice))
        gt_customer = ground_truth.get('customer_name', '')
        ai_customer = ai_prediction.get('customer_name', '')
        text_fields.append((gt_customer, ai_customer))
        
        gt_items = ground_truth.get('items', [])
        ai_items = ai_prediction.get('items', [])
        max_rows_to_check = max(len(gt_items), len(ai_items))
        
        for i in range(max_rows_to_check):
            # Numeric QTY
            gt_qty = str(gt_items[i].get('qty', '')) if i < len(gt_items) else ""
            ai_qty = str(ai_items[i].get('qty', '')) if i < len(ai_items) else ""
            numeric_fields.append((gt_qty, ai_qty))
            
            # Text Description
            gt_desc = str(gt_items[i].get('description', '')) if i < len(gt_items) else ""
            ai_desc = str(ai_items[i].get('description', '')) if i < len(ai_items) else ""
            text_fields.append((gt_desc, ai_desc))

        for gt_val, ai_val in numeric_fields:
            clean_gt = text_normalization(gt_val)
            clean_ai = text_normalization(ai_val)
            
            if clean_gt == "empty" and clean_ai == "empty":
                continue 
                
            total_fields_scored += 1
            total_numeric_fields += 1  
            
            if clean_gt == clean_ai:
                total_exact_matches += 1
                
            cer = jiwer.cer(clean_gt, clean_ai)
            accuracy = max(0, (1 - cer) * 100)
            total_accuracy_score += accuracy
            
        for gt_val, ai_val in text_fields:
            clean_gt = text_normalization(gt_val)
            clean_ai = text_normalization(ai_val)
            
            if clean_gt == "empty" and clean_ai == "empty":
                continue 
                
            total_fields_scored += 1
            
            precision_accuracy = calculate_word_precision(clean_gt, clean_ai)
            total_accuracy_score += precision_accuracy
            
    avg_total_accuracy = (total_accuracy_score / total_fields_scored) if total_fields_scored > 0 else 0
    avg_exact_match = (total_exact_matches / total_numeric_fields * 100) if total_numeric_fields > 0 else 0
    
    return avg_total_accuracy, avg_exact_match

# Run Model
actual_accuracy, actual_match = evaluate_model("actual")
rewritten_accuracy, rewritten_match = evaluate_model("rewritten")

print("\nFinal Evaluation:")
report_table = [
    ["3rd Ply (Actual Copies)", f"{actual_accuracy:.2f}%", f"{actual_match:.1f}%"],
    ["1st Ply Simulation (Darken Ink)", f"{rewritten_accuracy:.2f}%", f"{rewritten_match:.1f}%"]
]
print(tabulate(report_table, headers=["Document Type", "Average Accuracy (CER + Precision)", "Exact Field Match"], tablefmt="grid"))


Evaluating: ACTUAL COPIES
[1/20] Extracting CNR No 13092...
[2/20] Extracting CNR No 13086...
[3/20] Extracting CNR No 13087...
[4/20] Extracting CNR No 13088...
[5/20] Extracting CNR No 13178...
[6/20] Extracting CNR No 13195...
[7/20] Extracting CNR No 13182...
[8/20] Extracting CNR No 13089...
[9/20] Extracting CNR No 13097...
[10/20] Extracting CNR No 13172...
[11/20] Extracting CNR No 12712...
[12/20] Extracting CNR No 13193...
[13/20] Extracting CNR No 13094...
[14/20] Extracting CNR No 13196...
[15/20] Extracting CNR No 13194...
[16/20] Extracting CNR No 13096...
[17/20] Extracting CNR No 13093...
[18/20] Extracting CNR No 13176...
[19/20] Extracting CNR No 13177...
[20/20] Extracting CNR No 13098...

Evaluating: REWRITTEN COPIES
[1/20] Extracting CNR No 13092...
[2/20] Extracting CNR No 13086...
[3/20] Extracting CNR No 13087...
[4/20] Extracting CNR No 13088...
[5/20] Extracting CNR No 13178...
[6/20] Extracting CNR No 13195...
[7/20] Extracting CNR No 13182...
[8/20] Extract